# Mixed-table generation with TabFORGE

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SilenceX12138/TabFORGE/blob/master/docs/tutorial/notebooks/generation/01_unconditional_generation.ipynb)

> **Colab setup:** Open this notebook with the badge, then follow the [tutorial setup guide](https://silencex12138.github.io/TabFORGE/#/tutorials.md) to install `tabforge-ml[tutorials]` from PyPI in this runtime. Select a GPU for pretrained workloads.

## 1. Overview

This tutorial fine-tunes the Hugging Face-hosted `tabforge-v1` backbone on German Credit, generates a synthetic
mixed table, and evaluates marginal and pairwise fidelity. The Structure-aware Feature Encoder supplies pretrained
embeddings; TabFORGE owns the reconstruction and diffusion workflow.

In [ ]:
%pip install "tabforge-ml[tutorials]"

## 2. Imports and setup

The default path uses the Hugging Face-hosted TabFORGE backbone and the pretrained Structure-aware Feature Encoder.
Environment variables only control tutorial runtime and device placement.

In [ ]:
import os
import tempfile
from pathlib import Path

from tabforge import (
    MODEL_ID,
    TabFORGEEmbeddingConfig,
    TabFORGERuntimeConfig,
    TabFORGETrainingConfig,
)

RANDOM_STATE = 7
MAX_TRAIN_ROWS = int(os.environ.get("TABFORGE_TUTORIAL_MAX_ROWS", "512"))
MAX_STEPS = int(os.environ.get("TABFORGE_TUTORIAL_MAX_STEPS", "10000"))
DEVICE = os.environ.get("TABFORGE_TUTORIAL_DEVICE", "auto")

assert MODEL_ID == "tabforge-v1"

## 3. Prepare a real-world dataset

`TabularDataset` downloads German Credit from OpenML, keeps the mixed numerical/categorical schema,
and performs a reproducible stratified split. The tutorial caps the training rows to keep execution
short; increase `TABFORGE_TUTORIAL_MAX_ROWS` for a larger run.

In [ ]:
from tabcamel.data.dataset import TabularDataset

dataset = TabularDataset("credit-g", task_type="classification")
split = dataset.split("stratified", train_size=0.8, random_state=RANDOM_STATE)
train_set = split["train_set"]
test_set = split["test_set"]

X_train = train_set.X_df.iloc[:MAX_TRAIN_ROWS].reset_index(drop=True)
y_train = train_set.y_s.iloc[:MAX_TRAIN_ROWS].reset_index(drop=True)
categorical_features = tuple(dataset.categorical_feature_list)

print({"train": X_train.shape, "target": y_train.name})

## 4. Configure TabFORGE

Two embedding folds provide held-out training embeddings. `MAX_STEPS` controls the optimizer
budget; the tutorial defaults are an API demonstration and are not a benchmark protocol.

In [ ]:
from tabforge import TabFORGEGenerator

embedding_config = TabFORGEEmbeddingConfig(model_path="auto", n_folds=2)
training_config = TabFORGETrainingConfig(max_steps=MAX_STEPS, batch_size=128)
runtime_config = TabFORGERuntimeConfig(device=DEVICE)

generator = TabFORGEGenerator(
    task="classification",
    categorical_features=categorical_features,
    embedding_config=embedding_config,
    training_config=training_config,
    runtime_config=runtime_config,
    random_state=RANDOM_STATE,
)

## 5. Fit

`checkpoint="pretrained"` downloads and fine-tunes the pinned official TabFORGE backbone.

In [ ]:
generator.fit(
    X_train,
    y_train,
    checkpoint="pretrained",
)

## 6. Generate

The supervised generator returns features and the target separately.

In [ ]:
X_synthetic, y_synthetic = generator.generate(len(X_train), random_state=RANDOM_STATE + 1)
synthetic = X_synthetic.copy()
synthetic["target"] = y_synthetic

real = X_train.copy()
real["target"] = y_train.to_numpy()
synthetic.head()

## 7. Inspect results

TabEval's low-order density report exposes **Shape** (column distributions) and **Trend** (pairwise
relationships). Both scores range from zero to one, where higher is better.

In [ ]:
from tabeval.metrics.eval_density import LowOrderMetrics
from tabeval.plugins.core.dataloader import GenericDataLoader

metadata = {"columns": {column: dataset.col2type[column].sdmetrics_dtype for column in real.columns}}
density = LowOrderMetrics().evaluate(
    GenericDataLoader(real),
    GenericDataLoader(synthetic),
    metadata=metadata,
)
print({"Shape": density["shape"], "Trend": density["trend"]})

## 8. Save and load

A fitted generation checkpoint retains the context required for generation.

In [ ]:
with tempfile.TemporaryDirectory() as directory:
    checkpoint_path = generator.save_checkpoint(Path(directory))
    restored = TabFORGEGenerator.restore_from_checkpoint(checkpoint_path, device=DEVICE)
    restored_sample = restored.generate(5, random_state=RANDOM_STATE + 2)

print({"checkpoint": checkpoint_path.name, "restored_rows": len(restored_sample[0])})

## 9. Optional advanced configuration

Increase `MAX_STEPS` for a real training run, pass more embedding folds for stronger leakage control,
or tune diffusion sampling through `TabFORGEDiffusionConfig`. Keep evaluation on a held-out real split.